# Phase 09 — Threat Intelligence and Alert Correlation

This notebook reviews saved actual CIC-IDS2017 results from run 20261103T145019Z. It uses the reusable project modules and does not train or change either model. Phase 06 binary classification is unavailable. Friday ground-truth labels are excluded from detection input. The threat score is a heuristic prioritization value, not a probability of compromise.


## 1. Project introduction

Phase 09 combines supervised multi-class and unsupervised anomaly outputs into traceable flow events and alerts. Alerts summarize model evidence; they do not confirm an attack succeeded.


## 2. Threat intelligence architecture

Schemas, normalization, scoring, severity, event generation, correlation, alert generation, reporting, and orchestration live under src/intrusion_intelligence/intelligence. The service reuses persisted Phase 07 and Phase 08 batch predictors.


## 3. Loading source flow records

This preview selects predictive features and source provenance. Friday label and source-label columns are not read.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
RUN_ID = "20261103T145019Z"
RUN_DIR = ROOT / "reports" / "intelligence" / RUN_ID
FIGURE_DIR = ROOT / "reports" / "figures" / "intelligence" / RUN_ID
SOURCE = ROOT / "data/processed/CIC-IDS2017-phase04-v2/cleaned/extracted/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.parquet"
source_columns = pq.ParquetFile(SOURCE).schema_arrow.names
contract = json.loads((ROOT / "configs/feature_schema.json").read_text(encoding="utf-8"))
feature_names = [item["name"] for item in contract["features"] if item["role"] == "predictive" and item["included"]]
read_columns = [name for name in feature_names if name in source_columns]
read_columns += [name for name in ("__record_id", "__source_file", "__source_row_number") if name in source_columns]
source_batch = next(pq.ParquetFile(SOURCE).iter_batches(batch_size=5, columns=read_columns))
source_preview = source_batch.to_pandas()
print("Target columns selected:", sorted({"label", "__source_label"} & set(read_columns)))
source_preview


## 4. Loading or generating detector outputs

The following cell loads actual saved flow events produced by the Phase 09 CLI. Raw outputs preserve original detector values. Binary is unavailable.


In [ ]:
events = [json.loads(line) for line in (RUN_DIR / "flow_detection_events.jsonl").read_text(encoding="utf-8").splitlines()]
summary = json.loads((RUN_DIR / "threat_intelligence_summary.json").read_text(encoding="utf-8"))
sample_event = events[0]
sample_event["detector_results"]


## 5. Detection evidence normalization

Normalized evidence records the interpretation and availability alongside each score. The anomaly percentile uses 456,752 BENIGN Thursday validation observations scored with the frozen Phase 08 model.


In [ ]:
pd.DataFrame([
    {"detector": name, "availability": item["availability_status"], "normalized_value": item["normalized_value"],
     "interpretation": item["interpretation"], "calibration_status": item["calibration_status"]}
    for name, item in sample_event["normalized_evidence"].items()
])


## 6. Detector availability analysis

Unavailable Phase 06 binary output is not counted as a benign prediction.


In [ ]:
summary["detector_availability"]


## 7. Threat scoring configuration

Default weights are binary 0.30, multiclass 0.25, and anomaly 0.45. Missing or invalid signals are excluded and available weights are renormalized. The supervised signals are treated as related evidence.


In [ ]:
from intrusion_intelligence.intelligence.config import load_intelligence_config

CONFIG_PATH = ROOT / "configs/intelligence.toml"
config = load_intelligence_config(CONFIG_PATH)
config["scoring"], config["severity"]


## 8. Threat score calculation

The heuristic score is 100 times the weighted sum of normalized evidence divided by available weights. It is not a calibrated probability.


In [ ]:
scores = pd.Series([event["threat_score"] for event in events if event["threat_score"] is not None], name="heuristic_threat_score")
scores.describe()


## 9. Score contribution inspection

Each event includes signal values, configured/effective weights, contributions, and unavailable reasons.


In [ ]:
pd.DataFrame(sample_event["score_breakdown"]["signal_breakdown"]).T[
    ["availability_status", "normalized_value", "configured_weight", "effective_weight", "contribution"]
]


## 10. Severity classification

Default bands are LOW 0–24, MEDIUM 25–49, HIGH 50–74, and CRITICAL 75–100. A missing score maps to UNSCORED.


In [ ]:
from intrusion_intelligence.intelligence.severity import assign_severity

values = [0, 24.9, 25, 49.9, 50, 74.9, 75, 100]
pd.DataFrame({"score": values, "severity": [assign_severity(value, config["severity"]["bands"]) for value in values]})


## 11. Flow-level event generation

Flow events retain source provenance and detector results. Status summarizes model evidence and is not ground truth.


In [ ]:
{key: sample_event[key] for key in ("event_id", "flow_record_reference", "source_filename", "source_row_number", "event_status", "threat_score", "severity", "validation_warnings")}


## 12. Detector disagreement analysis

Disagreements retain different model views instead of forcing one prediction.


In [ ]:
summary["detector_disagreement_counts"]


## 13. Correlation rule configuration

Rules require explicit network identifiers and a timestamp. Destination port alone is not a valid key.


In [ ]:
config["correlation"]


## 14. Time-window correlation

The actual run used five-minute windows. The Phase 04 export lacks timestamps and endpoint IPs, so it produced no correlation groups. Row order was not substituted for time.


In [ ]:
summary["correlation_audit"]


## 15. Alert aggregation

Single-flow alerts require score at least 50 and a positive detector. Correlated alerts require at least two events and maximum score at least 25.


In [ ]:
alerts = json.loads((RUN_DIR / "security_alerts.json").read_text(encoding="utf-8"))
pd.DataFrame([{"alert_id": a["alert_id"], "type": a["alert_type"], "severity": a["severity"],
               "score": a["threat_score"], "flows": a["related_flow_count"],
               "categories": a["attack_categories"]} for a in alerts]).head(10)


## 16. Alert deduplication

Volatile inference timestamps remain in raw detector outputs but do not affect event fingerprints. Exact repeated alert fingerprints are merged with event references retained.


In [ ]:
{"generated_alerts": len(alerts), "duplicates_suppressed": summary["duplicate_alerts_suppressed"],
 "deduplication_records": summary["deduplication_records"]}


## 17. Alert explanation inspection

Explanations state the detector outputs, score contributions, disagreements, and metadata limitations that led to an alert.


In [ ]:
next(alert for alert in alerts if alert["explanation"])


## 18. Threat intelligence summary

The summary separates actual model outputs from unavailable signals and labels.


In [ ]:
{key: summary[key] for key in ("total_analyzed_flows", "flows_with_usable_detection_evidence",
                                  "flow_events_by_status", "generated_alert_count",
                                  "alerts_by_severity", "alerts_by_predicted_attack_category",
                                  "correlated_group_count", "input_sample")}


## 19. Alert visualizations

Plots are generated only for information present in the run: score distribution, severity, and detector agreement. There is no time-series or source-IP figure because those fields are absent.


In [ ]:
from IPython.display import Image, display

for filename in ("threat_score_distribution.png", "alert_severity_distribution.png", "detector_agreement.png"):
    display(Image(filename=str(FIGURE_DIR / filename)))


## 20. End-to-end pipeline execution

The actual run used the same reusable service exposed by the CLI. Set RUN_NEW_EXPERIMENT to True to create another bounded run; the default loads this saved run.


In [ ]:
from intrusion_intelligence.intelligence.service import run_intelligence_experiment

RUN_NEW_EXPERIMENT = False
if RUN_NEW_EXPERIMENT:
    execution_summary = run_intelligence_experiment(CONFIG_PATH)["summary"]
else:
    execution_summary = summary
execution_summary["experiment_run_id"]


## 21. Findings and limitations

The 3,000-flow sample had 2,923 NORMAL, 20 MALICIOUS_PREDICTION, 56 ANOMALOUS, and 1 SUSPICIOUS event. It generated 76 individual alerts (75 HIGH, 1 CRITICAL) and zero correlated groups. The binary classifier is unavailable. The export lacks timestamps and endpoint identities. Friday labels were not loaded into detection inputs and were not used for scoring or alert generation. Predicted categories and heuristic scores are not confirmed ground truth. No Phase 10 work is performed here.
